# Modul ADV-08: Efficient Computer Vision & Edge AI Architecture
## Hands-on Laboratory: Integer Quantization, Structured Pruning, and Roofline Performance Modeling

**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Level:** L3 Advanced Curriculum / Research Track R06 Preparation

Laboratorium komputasi ini mengimplementasikan konsep optimasi model visi pada perangkat tepi (*edge devices*):
1. **Kuantisasi Afin Seragam (Uniform Affine INT8/INT4):** Penentuan faktor skala $S$ dan zero-point $Z$ presisi.
2. **Straight-Through Estimator (STE):** Pelatihan sadar kuantisasi (*Quantization-Aware Training* / QAT).
3. **Pemangkasan Terstruktur (*Structured Channel Pruning*):** Evaluasi norma $L_1$ dan eliminasi kanal filter berlebih.
4. **Model Kinerja Roofline:** Klasifikasi *memory-bandwidth bound* vs *compute bound* pada akselerator tepi.
5. **Analisis Profil Latensi Percentile:** Estimasi P50, P95, dan P99 untuk verifikasi *real-time SLA*.

In [ ]:
import numpy as np

print("NumPy Version:", np.__version__)
print("Environment initialized successfully for ADV-08 Edge AI & Efficient CV Lab.")

In [ ]:
# 1. Kuantisasi Afin Seragam (Uniform Affine Quantization)
def compute_affine_quantization_params(r_min: float, r_max: float, bit_width: int = 8):
    q_min = 0
    q_max = (1 << bit_width) - 1
    clamped_min = min(r_min, 0.0)
    clamped_max = max(r_max, 0.0)
    if clamped_max == clamped_min:
        return 1.0, 0
    scale = (clamped_max - clamped_min) / float(q_max - q_min)
    zero_point = int(np.clip(np.round(-clamped_min / scale), q_min, q_max))
    return scale, zero_point

def quantize_affine(val, scale: float, zero_point: int, bit_width: int = 8):
    q_max = (1 << bit_width) - 1
    return np.clip(np.round(val / scale) + zero_point, 0, q_max).astype(np.int32)

def dequantize_affine(q_val, scale: float, zero_point: int):
    return scale * (q_val - zero_point)

# Uji coba parameter kuantisasi
r_min, r_max = -1.2, 2.6
scale, zp = compute_affine_quantization_params(r_min, r_max, bit_width=8)
print(f"Skala S: {scale:.6f}, Zero-Point Z: {zp}")

test_vals = np.array([-1.2, -0.5, 0.0, 1.45, 2.6])
q_vals = quantize_affine(test_vals, scale, zp, bit_width=8)
deq_vals = dequantize_affine(q_vals, scale, zp)

for orig, q, deq in zip(test_vals, q_vals, deq_vals):
    print(f"Orig: {orig:5.2f} -> Q: {q:3d} -> Deq: {deq:6.3f} | Error: {abs(orig - deq):.5f}")

In [ ]:
# 2. Straight-Through Estimator (STE) Forward & Backward
def ste_quantization(weights, scale, zp, q_min=0, q_max=255):
    q = np.clip(np.round(weights / scale) + zp, q_min, q_max)
    deq = scale * (q - zp)
    return deq

def ste_backward_mask(weights, r_min, r_max):
    # Gradien diteruskan jika bobot berada dalam rentang dinamis clipping
    return (weights >= r_min) & (weights <= r_max)

np.random.seed(42)
sample_weights = np.random.normal(0.0, 1.0, 10)
q_sim = ste_quantization(sample_weights, scale, zp)
mask = ste_backward_mask(sample_weights, r_min, r_max)

print("Bobot Asli:", np.round(sample_weights, 3))
print("Bobot Rekonstruksi (QAT forward):", np.round(q_sim, 3))
print("STE Pass Gradient Mask:", mask.astype(int))

In [ ]:
# 3. Pemangkasan Kanal Terstruktur (Structured Channel Pruning)
def structured_channel_pruning(weight_tensor: np.ndarray, target_sparsity: float = 0.5):
    C_out = weight_tensor.shape[0]
    k_prune = int(np.floor(C_out * target_sparsity))
    
    # Hitung L1-norm untuk setiap kanal keluaran
    norms = np.array([np.sum(np.abs(weight_tensor[i])) for i in range(C_out)])
    sorted_indices = np.argsort(norms)
    pruned_indices = sorted_indices[:k_prune]
    kept_indices = sorted_indices[k_prune:]
    
    return kept_indices, pruned_indices, norms

conv_weights = np.random.randn(8, 4, 3, 3)
kept, pruned, channel_norms = structured_channel_pruning(conv_weights, target_sparsity=0.5)

print("Norma Kanal:", np.round(channel_norms, 3))
print("Kanal Dipertahankan:", kept)
print("Kanal Dipangkas:", pruned)

In [ ]:
# 4. Evaluasi Model Roofline (Roofline Performance Model)
def evaluate_roofline(intensity: float, peak_gflops: float, peak_bw_gbs: float):
    knee = peak_gflops / peak_bw_gbs if peak_bw_gbs > 0 else 0.0
    mem_limit = intensity * peak_bw_gbs
    attainable = min(peak_gflops, mem_limit)
    regime = "memory-bound" if intensity < knee else "compute-bound"
    return attainable, regime, knee

# Profil perangkat tepi NVIDIA Jetson Nano: 40 GFLOPs, 25.6 GB/s
peak_compute = 40.0   # GFLOPs
peak_bandwidth = 25.6 # GB/s

dw_conv_intensity = 0.8  # Depthwise Separable Conv (FLOPs/Byte)
dense_intensity = 8.0     # Dense 1x1 Conv (FLOPs/Byte)

perf_dw, reg_dw, knee = evaluate_roofline(dw_conv_intensity, peak_compute, peak_bandwidth)
perf_dense, reg_dense, _ = evaluate_roofline(dense_intensity, peak_compute, peak_bandwidth)

print(f"Roofline Knee Point: {knee:.2f} FLOPs/Byte")
print(f"Depthwise Conv (I = {dw_conv_intensity}): Attainable {perf_dw:.2f} GFLOPs | Rezim: {reg_dw}")
print(f"Dense Conv (I = {dense_intensity}):       Attainable {perf_dense:.2f} GFLOPs | Rezim: {reg_dense}")

In [ ]:
# 5. Profiling Latensi Percentile (P50, P95, P99)
latencies = np.array([10.2, 11.5, 10.8, 12.1, 11.0, 10.9, 14.2, 28.5, 11.1, 10.7,
                      10.5, 11.2, 10.9, 11.8, 11.3, 11.0, 13.5, 25.0, 11.2, 10.8])

p50 = float(np.percentile(latencies, 50))
p95 = float(np.percentile(latencies, 95))
p99 = float(np.percentile(latencies, 99))
mean_lat = float(np.mean(latencies))

print(f"Rata-rata: {mean_lat:.2f} ms")
print(f"Median (P50): {p50:.2f} ms")
print(f"Tail Latency (P95): {p95:.2f} ms")
print(f"Worst-case (P99): {p99:.2f} ms")

### Kesimpulan & Rekomendasi Deployment
1. **Kuantisasi INT8:** Memberikan kompresi memori bobot sebesar $4\times$ dibandingkan FP32 dengan degradasi akurasi $<0.5\%$.
2. **Straight-Through Estimator (STE):** Menjamin konvergensi stabil pada pelatihan sadar kuantisasi (QAT) dengan meneruskan gradien identitas.
3. **Roofline Model Awareness:** Operasi konvolusi kedalaman (*depthwise*) dibatasi oleh bandwidth memori (*memory-bound*), sehingga fusi layer (*layer fusion*) dan perbaikan pola akses memori lebih efektif daripada peningkatan kapasitas komputasi akselerator.